In [1]:
!pip install timezonefinder
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from timezonefinder import TimezoneFinder
import pvlib
from pvlib.location import Location

In [5]:
df = pd.read_excel(r"D:\IIT BOMBAY\May 2025\Solcast_states_data\New_POA.xlsx")

In [6]:
df.head()

,timestamp,GHI,DNI,DHI,T2M,WS50M
0,2020-01-01 00:00:00,0.0,0.0,0.0,9.417958,2.832875
1,2020-01-01 01:00:00,0.0,0.0,0.0,9.118750,2.838771
2,2020-01-01 02:00:00,0.0,0.0,0.0,8.839271,2.822521
3,2020-01-01 03:00:00,0.0,0.0,0.0,8.668125,2.829854
4,2020-01-01 04:00:00,0.0,0.0,0.0,8.519313,2.831833


In [7]:
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%d-%m-%Y %H:%M')
df.index = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M')
df.head()
print(df)

                            timestamp  GHI  DNI  DHI        T2M     WS50M
timestamp                                                                
2020-01-01 00:00:00  01-01-2020 00:00  0.0  0.0  0.0   9.417958  2.832875
2020-01-01 01:00:00  01-01-2020 01:00  0.0  0.0  0.0   9.118750  2.838771
2020-01-01 02:00:00  01-01-2020 02:00  0.0  0.0  0.0   8.839271  2.822521
2020-01-01 03:00:00  01-01-2020 03:00  0.0  0.0  0.0   8.668125  2.829854
2020-01-01 04:00:00  01-01-2020 04:00  0.0  0.0  0.0   8.519313  2.831833
...                               ...  ...  ...  ...        ...       ...
2023-12-31 18:59:00  31-12-2023 18:59  0.0  0.0  0.0  13.940438  2.624312
2023-12-31 19:59:00  31-12-2023 19:59  0.0  0.0  0.0  13.384146  2.558021
2023-12-31 20:59:00  31-12-2023 20:59  0.0  0.0  0.0  12.966604  2.417104
2023-12-31 21:59:00  31-12-2023 21:59  0.0  0.0  0.0  12.570417  2.270438
2023-12-31 22:59:00  31-12-2023 22:59  0.0  0.0  0.0  12.169729  2.186104

[35040 rows x 6 columns]


In [11]:
tilt = 21.14
azimuth = 180

location_latitude =  21.14
location_longitude = 79.08

location = Location(latitude=location_latitude, longitude=location_longitude)
tf = TimezoneFinder()
tz=tf.certain_timezone_at(lat=location_latitude, lng=location_longitude)
print(tz)

df.index = df.index.tz_localize(tz)

loc = pvlib.location.Location(location_latitude, location_longitude, tz)

sun = loc.get_solarposition(df.index)

Asia/Kolkata


In [15]:
s = pd.merge(df, sun, left_on=df.index, right_on=sun.index)
s.head()

,key_0,timestamp,GHI,DNI,DHI,T2M,WS50M,apparent_zenith,zenith,apparent_elevation,elevation,azimuth,equation_of_time
0,2020-01-01 00:00:00+05:30,01-01-2020 00:00,0.0,0.0,0.0,9.417958,2.832875,175.684168,175.684168,-85.684168,-85.684168,242.575796,-2.972316
1,2020-01-01 01:00:00+05:30,01-01-2020 01:00,0.0,0.0,0.0,9.118750,2.838771,169.783871,169.783871,-79.783871,-79.783871,102.914770,-2.992228
2,2020-01-01 02:00:00+05:30,01-01-2020 02:00,0.0,0.0,0.0,8.839271,2.822521,156.026280,156.026280,-66.026280,-66.026280,99.464696,-3.012131
3,2020-01-01 03:00:00+05:30,01-01-2020 03:00,0.0,0.0,0.0,8.668125,2.829854,142.252246,142.252246,-52.252246,-52.252246,100.763611,-3.032026
4,2020-01-01 04:00:00+05:30,01-01-2020 04:00,0.0,0.0,0.0,8.519313,2.831833,128.569506,128.569506,-38.569506,-38.569506,103.231076,-3.051912


In [17]:
s = s.drop(columns=['key_0','apparent_zenith', 'apparent_elevation', 'elevation', 'equation_of_time'], axis=1)
s.head()

,timestamp,GHI,DNI,DHI,T2M,WS50M,zenith,azimuth
0,01-01-2020 00:00,0.0,0.0,0.0,9.417958,2.832875,175.684168,242.575796
1,01-01-2020 01:00,0.0,0.0,0.0,9.118750,2.838771,169.783871,102.914770
2,01-01-2020 02:00,0.0,0.0,0.0,8.839271,2.822521,156.026280,99.464696
3,01-01-2020 03:00,0.0,0.0,0.0,8.668125,2.829854,142.252246,100.763611
4,01-01-2020 04:00,0.0,0.0,0.0,8.519313,2.831833,128.569506,103.231076


In [19]:
df1 = pvlib.irradiance.erbs(s['GHI'], s['zenith'], s.index)

s['beam'] = pvlib.irradiance.beam_component(tilt, azimuth, s['zenith'], s['azimuth'], s['DNI'])
s['sky'] = pvlib.irradiance.isotropic(tilt, s['DHI'])
s['poa'] = s['beam']+s['sky']

In [21]:
s.to_csv(r"D:\IIT BOMBAY\May 2025\Solcast_states_data\POA_2020_2030.csv", index=False)